# Notebook 03 — Results Analysis & Statistical Testing

Loads saved experiment results from `results/tables/main_results.csv` and produces
publication-ready visualisations and statistical comparisons.

**Prerequisites:** run `experiments/run_experiment.py` first.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

from qml_hybrid.evaluate import (
    compute_metrics,
    generate_results_table,
    mcnemar_test,
)
from qml_hybrid.utils import load_dataset, plot_results_heatmap

# Load test labels
_, _, _, _, _, y_test = load_dataset(seed=42)
print(f"Test set: {len(y_test)} samples | class balance: {np.bincount(y_test)}")

# Try loading saved results
results_path = Path("../results/tables/main_results.csv")
if results_path.exists():
    df_saved = pd.read_csv(results_path, index_col=0)
    print(f"\nLoaded saved results from {results_path}")
    print(df_saved.to_string())
else:
    print(f"\n⚠  {results_path} not found. Run experiments/run_experiment.py first.")
    print("   Running a quick demo with random predictions for illustration...")
    rng = np.random.default_rng(42)
    demo_preds = rng.integers(0, 2, size=len(y_test))
    demo_probs = rng.dirichlet([1, 1], size=len(y_test))
    df_saved = generate_results_table(
        y_test,
        {"VQC (demo)": {"preds": demo_preds, "probs": demo_probs}},
    )
    print(df_saved.to_string())


## Statistical Significance Testing

We use **McNemar's test** (McNemar, 1947) for paired comparison of binary classifiers.

**Why McNemar's, not a t-test?**
- Classifier errors are binary (correct/incorrect per sample), not continuous
- The same test set is used for all classifiers → observations are *paired*
- McNemar's is distribution-free (non-parametric) — appropriate here
- We apply **Bonferroni correction** for multiple comparisons (comparing VQC against 3 baselines → α_corrected = 0.05/3 ≈ 0.017)

The null hypothesis H₀: the two classifiers have the same error rate.
Reject H₀ when Bonferroni-corrected p < 0.05.

In [ ]:
# Heatmap of numeric metrics across all models
numeric_cols = df_saved.select_dtypes(include=[float, int]).columns.tolist()
fig, ax = plt.subplots(figsize=(len(numeric_cols) * 1.6 + 1, len(df_saved) * 0.9 + 1))
sns.heatmap(
    df_saved[numeric_cols].astype(float),
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    ax=ax,
    vmin=0,
    vmax=1,
    linewidths=0.5,
)
ax.set_title("Model Comparison — Wisconsin Breast Cancer (test set)", pad=12)
plt.tight_layout()
plt.show()

print("\n" + "=" * 55)
print("Full results table:")
print("=" * 55)
print(df_saved.to_string())
